# Getting Started with Automated-LLM-Probes

In [5]:
from __future__ import annotations
import os,time,tqdm,pandas as pd
from pathlib import Path
import automated_intelligence_tests as ait
import automated_llm_probes as alp
assert os.environ.get("OPENAI_API_KEY")

### Models available

In [39]:
models = alp.load_models()
print(f"|Models available: {len(models)}|\n")
for m in models:
    print(f"  {m['name'][:20]:20s} {m['vendor'][:11]:12s} \
{m['api'][:11]:12s} {m['model_id'][:12]:14s} {m['status'][:12]:12s}")

|Models available: 91|

  grok-code-fast       xai          spacexai     grok-code-fa   ok          
  grok-2               xai          spacexai     grok-2                     
  grok-4               xai          spacexai     grok-4         ok          
  grok-4.2             xai          spacexai     grok-4.20-03   ok          
  grok-4.2-reasoning   xai          spacexai     grok-4.20-03   ok          
  grok-4.3             xai          spacexai     grok-4.3       ok          
  grok-4.5             xai          spacexai     grok-4.5       ok          
  grok-build-0.1       xai          spacexai     grok-build-0   ok          
  grok-4.6             xai          spacexai     grok-4.6       ok          
  grok-4.2-multi-agent xai          spacexai     grok-4.20-mu   failed      
  hunyuan-3            tencent      hunyuan      hy3            ok          
  hunyuan-lite         tencent      hunyuan      hunyuan-lite   failed      
  qwen-turbo           qwen         qwen         qwe

### Probe models

In [30]:
def probe_models(model_rows,prompt=[{"role":"user","content":"reply ok"}]):
    retries = alp.MAX_RETRIES; alp.MAX_RETRIES = 1
    try:
        for model in tqdm.tqdm(model_rows):
            start = time.time()
            try:
                model['reply'] = alp.call_model(model,prompt)
                model["status"] = "ok" if model['reply'].strip() else "failed"
                model["errors"] = None
            except Exception as e:
                model['reply'] = None
                model["status"] = 'failed'
                model["errors"] = str(e).lower()
            model["speed_per_call"] = f"{round((time.time()-start),2)} seconds"
    finally:
        alp.MAX_RETRIES = retries
    return model_rows

models = alp.load_models()
probes = probe_models(models)
probes = pd.DataFrame(probes).set_index('name')

100%|████████████████████████████████████████████████████████████████████| 90/90 [03:03<00:00,  2.04s/it]


In [32]:
probes.to_csv('./models.csv')
probes.head()

,vendor,api,model_id,release_date,temperature,speed_per_call,status,reply,errors
name,,,,,,,,,
grok-code-fast,xai,spacexai,grok-code-fast,26-Aug-25,0.5,6.31 seconds,ok,ok,None
grok-4,xai,spacexai,grok-4,9-Jul-25,0.5,2.55 seconds,ok,ok,None
grok-4.2,xai,spacexai,grok-4.20-0309-non-reasoning,9-Mar-26,0.5,1.54 seconds,ok,ok,None
grok-4.2-reasoning,xai,spacexai,grok-4.20-0309-reasoning,9-Mar-26,0.5,1.72 seconds,ok,ok,None
grok-4.3,xai,spacexai,grok-4.3,30-Apr-26,0.5,2.19 seconds,ok,ok,None


### Probed tasks

In [ ]:
path = Path('./data/'); n=1
print(f"|Probed tasks: {len([p for p in path.iterdir() if p.is_dir()])}|\n")
for d in path.iterdir():
    if (d.is_dir()) and ('.' not in d.name):
        count = sum(1 for f in d.rglob('*') if f.is_file())
        print(f"  {n}. {d.name.upper()[:7]:8}:  {count}"); n+=1

### Tests availabe

In [4]:
ait_counts = ait.list_available_tests()
print(f'|Available tests: {len(ait_counts)}|\n')
for i,v in ait_counts.items():
    print(f'  {i} : {v}')

|Available tests: 4|

  AUT : Alternative Uses Task
  CAT : Convergent Association Task
  CWT : Creative Writing Task
  DAT : Divergent Association Task


### Trial-run

In [44]:
models_to_try = ['claude-haiku-4.5',
 'claude-opus-4.5',
 'claude-opus-4.7',
 'claude-opus-5',
 'claude-sonnet-4.5',
 'gpt-3.5-turbo',
 'gpt-4-turbo',
 'gpt-4-turbo',
 'gpt-4o',
 'gpt-4o-mini',
 'gpt-5.4',
 'grok-4.2',
 'grok-4.3',
 'grok-4.5',
 'grok-build-0.1',
 'llama-3.1-8b',
 'llama-3.2-3b',
 'llama-4-guard-12b',
 'llama-4-maverick',
 'llama-4-scout']

cues_to_try = ["brick", "paperclip"]

models = alp.ready_models()
models_to_try = [m for m in models if m["name"] in models_to_try]
sorted([m["name"] for m in models_to_try])

['claude-haiku-4.5',
 'claude-opus-4.5',
 'claude-opus-4.7',
 'claude-opus-5',
 'claude-sonnet-4.5',
 'gpt-3.5-turbo',
 'gpt-4-turbo',
 'gpt-4-turbo',
 'gpt-4o',
 'gpt-4o-mini',
 'gpt-5.4',
 'grok-4.2',
 'grok-4.3',
 'grok-4.5',
 'grok-build-0.1',
 'llama-3.1-8b',
 'llama-3.2-3b',
 'llama-4-guard-12b',
 'llama-4-maverick',
 'llama-4-scout']

In [45]:
alp.collect("AUT", models=models_to_try, n_per_model=300)

  grok-4.2: 620/300 done — skip
  grok-4.3: 620/300 done — skip
  grok-4.5: 620/300 done — skip
  grok-build-0.1: 620/300 done — skip
  gpt-3.5-turbo: 731/300 done — skip
  gpt-4-turbo: 620/300 done — skip
  gpt-4o: 681/300 done — skip
  gpt-5.4: 618/300 done — skip
  gpt-4o-mini: 620/300 done — skip
  gpt-4-turbo: 620/300 done — skip
  llama-4-guard-12b: 600/300 done — skip
  llama-4-scout: 620/300 done — skip
  llama-4-maverick: 600/300 done — skip
  llama-3.2-3b: 600/300 done — skip
  llama-3.1-8b: 600/300 done — skip
  claude-sonnet-4.5: 627/300 done — skip
  claude-haiku-4.5: 620/300 done — skip
  claude-opus-4.5: 620/300 done — skip
  claude-opus-4.7: 600/300 done — skip
  claude-opus-5: 503/300 done — skip


### Load functions

In [130]:
from __future__ import annotations
import re, pandas as pd
import glove_word_embeddings as gwe

def parse_dat(raw):
    tokens = re.split(r"[,\n\r]+", str(raw or "").strip().strip("\"'"))
    nouns = [n for n in (gwe.pre.clean_word(t) for t in tokens) if n][:10]
    return nouns + [""] * (10 - len(nouns))

def parse_aut(raw):
    uses, text = [], re.sub(r"<br\s*/?>", "\n", str(raw or ""), flags=re.I)
    for line in re.split(r"[\n\r,;]+", text):
        line = re.sub(r"^\s*[\d\.\)\-]+\s*", "", line)
        if toks := [t for t in (gwe.pre.clean_word(t) for t in line.split()) if t]:
            uses.append(" ".join(toks))
    return ", ".join(uses)

def parse_cwt(raw):
    text = re.sub(r"^#+\s*.*$", "", str(raw or ""), flags=re.M)
    text = re.sub(r"^\s*Title:.*$", "", text, flags=re.M | re.I)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def parse_temperature(df):
    if "temperature" not in df.columns:
        df["temperature"] = df["temperature_std"]
    df["temperature"] = df["temperature"].replace("default", 0.5)
    return df.drop(columns=["temperature_std"], errors="ignore")

def parse_model_name(df):
    model_rename = pd.DataFrame(alp.load_models())
    model_rename = model_rename.set_index('model_id')['name'].to_dict()
    model_rename.update({'deepseek/deepseek-chat':'deepseek-2.5-chat',
                         'deepseek-chat':'deepseek-2.5-chat',
                         'deepseek/deepseek-r1':'deepseek-r1',
                         'deepseek/deepseek-v4-pro':'deepseek/deepseek-4-pro',
                         'gpt-4o-2024-11-20':'gpt-4o',
                         'gpt-5':'gpt-5',
                         'gpt-5-mini':'gpt-5-mini',
                         'gpt-5.4':'gpt-5.4',
                         'grok-code-fast-1':'grok-code-fast',
                         'moonshotai/kimi-k2':'kimi-k2',
                         'kimi-k2':'kimi-k2',
                         'llama4-maverick-instruct-basic':'llama-4-maverick',
                         'llama4-scout-instruct-basic':'llama4-4-scout'})
    df['model_name'] = df.apply(lambda x:model_rename[x['model_id']],axis=1)
    return df

def load_task(df, task):
    df = parse_temperature(df)
    df = parse_model_name(df)    
    if task == "dat":
        parsed = df["raw"].map(parse_dat)
        df[[f"noun_{i}" for i in range(10)]] = parsed.tolist()
        df["response_clean"] = parsed.map(lambda x: ", ".join(n for n in x if n))
        extra = [f"noun_{i}" for i in range(10)]
    elif task == "aut":
        df["object"] = df["prompt"].str.extract(
            r"object: (.+?)\?", expand=False).str.strip()
        df["response_clean"] = df["raw"].map(parse_aut)
        extra = ["object"]
    elif task == "cwt":
        parts = (df["prompt"].str.extract(
            r"words?(?:\(s\))?:\s*(.+?)\.", expand=False).fillna("").map(
            lambda s: [w.strip() for w in str(s).split(",") if w.strip()]))
        parts = parts.map(lambda xs: ["".join(xs)] if xs and max(map(len, xs)) == 1 else xs)
        df[["cue_0", "cue_1", "cue_2"]] = pd.DataFrame(
            parts.tolist(), index=df.index).reindex(columns=range(3))
        df["response_clean"] = df["raw"].map(parse_cwt)
        extra = ["cue_0", "cue_1", "cue_2"]
    else:
        raise ValueError(task)
    cols = ["task", "model_name", "model_id", "provider", 
            "rep", "temperature", *extra, "prompt", 
            "response_clean", "ts_utc", "hash"]
    return df[[c for c in cols if c in df.columns]].sort_values(
        ["model_name", "rep"]).reset_index(drop=True)

print("All functions loaded...")

All functions loaded...


### Parse & merge data

In [133]:
df = parse_model_name(df)

In [131]:
model_rename

{'grok-code-fast': 'grok-code-fast',
 'grok-2': 'grok-2',
 'grok-4': 'grok-4',
 'grok-4.20-0309-non-reasoning': 'grok-4.2',
 'grok-4.20-0309-reasoning': 'grok-4.2-reasoning',
 'grok-4.3': 'grok-4.3',
 'grok-4.5': 'grok-4.5',
 'grok-build-0.1': 'grok-build-0.1',
 'grok-4.6': 'grok-4.6',
 'grok-4.20-multi-agent-0309': 'grok-4.2-multi-agent',
 'hy3': 'hunyuan-3',
 'hunyuan-lite': 'hunyuan-lite',
 'qwen-turbo': 'qwen-turbo',
 'qwen3-235b-a22b-instruct-2507': 'qwen-3-235b-instruct',
 'qwen-max': 'qwen-max',
 'qwen-plus': 'qwen-plus',
 'qwen3.7-max': 'qwen-3.7-max',
 'qwen3.5-plus': 'qwen-3.5-plus',
 'qwen3-max-2026-01-23': 'qwen-3-max',
 'gpt-3.5-turbo': 'gpt-3.5-turbo',
 'gpt-4-turbo-2024-04-09': 'gpt-4-turbo',
 'gpt-4o-2024-08-06': 'gpt-4o',
 'o4-mini-2025-04-16': 'gpt-o4-mini',
 'gpt-4.1-2025-04-14': 'gpt-4.1',
 'gpt-4.1-mini-2025-04-14': 'gpt-4.1-mini',
 'gpt-4.1-nano-2025-04-14': 'gpt-4.1-nano',
 'gpt-5-2025-08-07': 'gpt-5',
 'gpt-5-mini-2025-08-07': 'gpt-5-mini',
 'gpt-5.1-2025-11-13'

In [132]:
for task in (
    "dat", 
    "aut",
    "cwt"
):
    print(f"Parsing {task.upper()}...")
    task = task.lower()
    df = pd.DataFrame.from_dict(alp.parse_and_merge(task),orient='index')
    df = load_task(df,task)
    print(df.shape,df.columns)
    df.to_csv(f"./data/{task.upper()}_AI_2026.csv", index=False)
    print('Saved...')

Parsing DAT...


dat: 100%|████████████████████████████████████████████████████████| 10778/10778 [00:17<00:00, 630.13it/s]


(10643, 19) Index(['task', 'model_name', 'model_id', 'provider', 'rep', 'temperature',
       'noun_0', 'noun_1', 'noun_2', 'noun_3', 'noun_4', 'noun_5', 'noun_6',
       'noun_7', 'noun_8', 'noun_9', 'prompt', 'response_clean', 'ts_utc'],
      dtype='object')
Saved...
Parsing AUT...


aut: 100%|████████████████████████████████████████████████████████| 13687/13687 [00:29<00:00, 471.94it/s]


(13665, 10) Index(['task', 'model_name', 'model_id', 'provider', 'rep', 'temperature',
       'object', 'prompt', 'response_clean', 'ts_utc'],
      dtype='object')
Saved...
Parsing CWT...


cwt: 100%|████████████████████████████████████████████████████████| 15753/15753 [01:08<00:00, 229.54it/s]


(15646, 12) Index(['task', 'model_name', 'model_id', 'provider', 'rep', 'temperature',
       'cue_0', 'cue_1', 'cue_2', 'prompt', 'response_clean', 'ts_utc'],
      dtype='object')
Saved...


In [134]:
raw = pd.DataFrame.from_dict(alp.parse_and_merge("dat"), orient="index")
print(raw.model_name.value_counts())
print(raw.loc[raw.model_name.eq(m),"temperature_std"].fillna("default").value_counts())

dat: 100%|████████████████████████████████████████████████████████| 10778/10778 [00:13<00:00, 778.03it/s]


model_name
Claude Opus 4.7      650
llama-4-guard-12b    450
Grok Build 0.1       400
Claude Haiku 4.5     400
Grok 4.2             400
Claude Opus 4.5      400
Grok 4.5             400
Grok 4.3             400
Claude Opus 5        400
Claude Sonnet 4.5    400
GPT-4o-mini          350
GPT-4o               350
Grok 4.6             350
GPT-4-Turbo          350
Llama-3.1 8b         350
Llama-4 Scout        350
GPT-3.5-Turbo        350
Llama-4 Maverick     350
Llama-3.2 3b         350
GPT-5.4              350
GPT-5.6-Sol          350
DeepSeek-Chat        275
DeepSeek-3.2         275
gpt-3.5-turbo        250
grok-4.3             200
grok-4.5             200
grok-build-0.1       200
llama-4-scout        100
gpt-5.4              100
gpt-4-turbo          100
gpt-4o-mini          100
gpt-4o               100
llama-3.1-8b         100
llama-3.2-3b         100
llama-4-maverick     100
claude-haiku-4.5      50
claude-opus-4.5       50
claude-opus-4.7       50
claude-opus-5         50
claude-sonnet-